In [1]:
import pandas as pd
import numpy as np

In [53]:
data = pd.read_csv('/home1/bingi' + '/mimic_iv_data.csv')
test = pd.read_csv('/home1/bingi' + '/mimic_iii_data.csv')
test.head()

,SUBJECT_ID,DOD,HADM_ID,ICUSTAY_ID,FIRST_CAREUNIT,LAST_CAREUNIT,INTIME,OUTTIME,LOS,age,...,mean_systolic_BP,mean_diastolic_BP,mean_mean_BP,mean_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,gender
0,115,2195-03-24,114585,232514,SICU,SICU,2194-10-17 13:22:28,2194-10-24 16:26:47,7.1280,75.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F
1,357,2201-08-02,117876,235292,SICU,SICU,2199-12-24 19:52:27,2200-01-07 21:24:42,14.0641,64.0,...,98.280000,57.400000,69.240000,100.000000,66.0,84.0,38.0,48.0,100.0,M
2,414,2124-05-02,153627,203468,SICU,SICU,2124-04-29 16:48:34,2124-05-02 08:29:58,2.6538,80.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,M
3,1339,2140-02-27,158998,248434,SICU,SICU,2138-10-11 23:38:05,2138-10-15 19:39:35,3.8344,67.0,...,97.923077,50.576923,60.884615,97.384615,73.0,84.0,37.0,52.0,94.0,F
4,1342,2107-03-19,178263,273691,SICU,SICU,2107-02-27 23:12:30,2107-03-19 15:14:40,19.6682,73.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,F


In [54]:
df = data.copy()

In [4]:
df.columns

Index(['subject_id', 'dod', 'hadm_id', 'stay_id', 'first_careunit',
       'last_careunit', 'intime', 'outtime', 'los', 'age',
       ...
       'max_systolic_BP', 'max_diastolic_BP', 'max_mean_BP', 'max_Spo2',
       'min_Heart Rate', 'min_systolic_BP', 'min_diastolic_BP', 'min_mean_BP',
       'min_Spo2', 'gender'],
      dtype='object', length=121)

In [55]:
threshold = len(df) * 0.7

# Drop columns where non-null count < threshold
df_cleaned = df.dropna(axis=1, thresh=threshold)

# Optional: see which columns were dropped
dropped_cols = set(df.columns) - set(df_cleaned.columns)
print("Dropped columns (more than 30% missing):", dropped_cols)

df = df_cleaned.copy()

len(dropped_cols)

Dropped columns (more than 30% missing): {'mean_Hemoglobin', 'min_Bilirubin', 'max_Hematocrit', 'mean_Hematocrit', 'insulin', 'max_Lactate', 'min_Albumin', 'max_Albumin', 'anticoagulant', 'max_Hemoglobin', 'dod', 'mean_Lactate', 'mean_AST', 'mean_Albumin', 'max_Troponin', 'mean_Bilirubin', 'min_Lactate', 'max_AST', 'min_Hematocrit', 'min_AST', 'aspirin', 'min_Hemoglobin', 'min_Troponin', 'max_CRP', 'mean_Troponin', 'max_Bilirubin', 'min_CRP', 'statin', 'mean_CRP'}


29

In [56]:
#'first_careunit', 'last_careunit', 'insurance', 'ethnicity', 'gender'

test = test.rename(columns={
    'FIRST_CAREUNIT': 'first_careunit',
    'LAST_CAREUNIT': 'last_careunit',
    'INSURANCE' : 'insurance',
    'ETHNICITY': 'ethnicity',
    'LOS': 'los'
    })
test.columns

Index(['SUBJECT_ID', 'DOD', 'HADM_ID', 'ICUSTAY_ID', 'first_careunit',
       'last_careunit', 'INTIME', 'OUTTIME', 'los', 'age',
       ...
       'mean_systolic_BP', 'mean_diastolic_BP', 'mean_mean_BP', 'mean_Spo2',
       'min_Heart Rate', 'min_systolic_BP', 'min_diastolic_BP', 'min_mean_BP',
       'min_Spo2', 'gender'],
      dtype='object', length=121)

In [57]:
rem = ['mean_Hemoglobin', 'min_Bilirubin', 'max_Hematocrit', 'mean_Hematocrit',
       'insulin', 'max_Lactate', 'min_Albumin', 'max_Albumin', 'anticoagulant', 
       'max_Hemoglobin', 'dod', 'mean_Lactate', 'mean_AST', 'mean_Albumin', 
       'max_Troponin', 'mean_Bilirubin', 'min_Lactate', 'max_AST', 'min_Hematocrit',
       'min_AST', 'aspirin', 'min_Hemoglobin', 'min_Troponin', 'max_CRP', 
       'mean_Troponin', 'max_Bilirubin', 'min_CRP', 'statin', 'mean_CRP', 'DOD'] 

test_cleaned = test.drop(columns=rem,errors='ignore')
test = test_cleaned.copy()
print(test_cleaned.shape)

#test = pd.DataFrame(test)
print(test.shape)

#x = [c for c in df.columns if c not in test.columns]
#print(x)


(681, 92)
(681, 92)


In [8]:
#df.info()

In [9]:
#test.info()

In [58]:
df['outtime'] = pd.to_datetime(df['outtime'])
df['intime'] = pd.to_datetime(df['intime'])

test['OUTTIME'] = pd.to_datetime(test['OUTTIME'])
test['INTIME'] = pd.to_datetime(test['INTIME'])



In [59]:
# Drop IDs and datetimes (models can’t use raw timestamps)

drop_cols = [c for c in ['subject_id','hadm_id','stay_id'] if c in df.columns]
df = df.drop(columns=drop_cols, errors='ignore')
dt_cols = df.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
df = df.drop(columns=list(dt_cols), errors='ignore')

#test set MIMIC III

drop_cols_test = [c for c in ['SUBJECT_ID','HADM_ID','ICUSTAY_ID'] if c in test.columns]
test = test.drop(columns=drop_cols_test, errors='ignore')
dt_cols = test.select_dtypes(include=['datetime64[ns]', 'datetime64']).columns
test = test.drop(columns=list(dt_cols), errors='ignore')

In [60]:
df.to_csv('mimic-iv-statstest.csv')
test.to_csv('mimic-iii-statstest.csv')

In [61]:
y = df['is_stroke'].astype(int)
df = df.drop(columns=['is_stroke'])

y_test = test['is_stroke'].astype(int)
X_test = test.drop(columns=['is_stroke'])

In [62]:
num_cols = df.select_dtypes(include=['float64','int64']).columns
cat_cols = df.select_dtypes(include=['object','category']).columns

test_num_cols = test.select_dtypes(include=['float64','int64']).columns
test_cat_cols = test.select_dtypes(include=['object','category']).columns

In [63]:
#df['dod'].dtype

In [15]:
#no missing values in category columns
#dealing missing values in numerical coumns with Iterative imputing

In [64]:
num_st = [c for c in num_cols if c not in ['pvd', 'htn', 'copd', 'diabetes',
                          'renal', 'liver', 'pud', 'sepsis', 'chf', 'cancer',
                          'rheumatic', 'Alcohol', 'Tobacco', 'Opioid',
                          'Cannabis', 'Cocaine', 'Sedative', 'Other', 'Smoking'] ]

test_num_st = [c for c in test_num_cols if c not in ['pvd', 'htn', 'copd', 'diabetes',
                          'renal', 'liver', 'pud', 'sepsis', 'chf', 'cancer',
                          'rheumatic', 'Alcohol', 'Tobacco', 'Opioid',
                          'Cannabis', 'Cocaine', 'Sedative', 'Other', 'Smoking'] ]

In [65]:
elix_weights = {
'chf': 7,
'pvd': 2,
'htn': -1,
'copd': 3,
'diabetes': 5,
'renal': 5,
'liver': 11,
'pud': 0,
'cancer': 14,
'rheumatic': 0,
'alcohol': 0,
'Tobacco': 0,
'Opiod': 0,
'Cannabis': 0,
'Cocaine': 0, 
'Sedative': 0, 
'Other': 0, 
'Smoking': 0,
'sepsis': 0
}

In [66]:
def calculate_elixhauser_score(data):
    score = np.zeros(len(data))
    for col, weight in elix_weights.items():
        if col in data.columns:
            score += data[col] * weight
    return score

df['elix_score'] = calculate_elixhauser_score(df)

X_test['elix_score'] = calculate_elixhauser_score(X_test)

In [67]:

X_test.isnull().sum()

first_careunit        0
last_careunit         0
los                   0
age                   0
insurance             0
                   ... 
min_diastolic_BP    150
min_mean_BP         150
min_Spo2            131
gender                0
elix_score            0
Length: 87, dtype: int64

In [68]:
def calculate_sofa(row):
    sofa = 0
    
    spo2 = row['mean_Spo2']

    if spo2 < 85:
        sofa += 4
    elif spo2 < 89:
        sofa += 3
    elif spo2 < 92:
        sofa += 2
    elif spo2 < 94:
        sofa += 1
    
    platelets = row['mean_Platelet Count']

    if platelets < 20:
        sofa += 4
    elif platelets < 50:
        sofa += 3
    elif platelets < 100:
        sofa += 2
    elif platelets < 150:
        sofa += 1

    map_bp = row['mean_mean_BP']

    if map_bp < 70:
        sofa += 1
    
    
    creat = row['mean_Creatinine']

    if creat > 5:
        sofa += 4
    elif creat >= 3.5:
        sofa += 3
    elif creat >= 2:
        sofa += 2
    elif creat >= 1.2:
        sofa += 1

    return sofa


# Apply to DataFrame
df['SOFA_score'] = df.apply(calculate_sofa, axis=1)

X_test['SOFA_score'] = X_test.apply(calculate_sofa, axis = 1)

In [19]:
X_test['SOFA_score'].head()

0    1
1    3
2    4
3    1
4    0
Name: SOFA_score, dtype: int64

In [69]:
num_cols = df.select_dtypes(include=['float64','int64']).columns

test_num_cols = X_test.select_dtypes(include=['float64','int64']).columns

In [70]:
#One hot encoding of cat cols

from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)

# Fit only on TRAIN
#encoder.fit(df[['cat_cols']])

df_encoded = encoder.fit_transform(df[cat_cols])
X_test_encoded  = encoder.transform(X_test[test_cat_cols])

cat_feature_names = encoder.get_feature_names_out(cat_cols)

df_encoded = pd.DataFrame(
    df_encoded,
    columns=cat_feature_names,
    index=df.index
)

cat_feature_names = encoder.get_feature_names_out(cat_cols)

X_test_encoded = pd.DataFrame(
    X_test_encoded,
    columns=cat_feature_names,
    index=X_test.index
)
#df = pd.get_dummies(df, columns=cat_cols, drop_first=True)
X_test_encoded.head()

,first_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Cardiac Vascular Intensive Care Unit (CVICU),last_careunit_Coronary Care Unit (CCU),last_careunit_Medical Intensive Care Unit (MICU),last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),last_careunit_Neuro Intermediate,last_careunit_Neuro Stepdown,last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),last_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Trauma SICU (TSICU),...,ethnicity_AMERICAN INDIAN/ALASKA NATIVE,ethnicity_ASIAN,ethnicity_BLACK/AFRICAN AMERICAN,ethnicity_HISPANIC/LATINO,ethnicity_OTHER,ethnicity_UNABLE TO OBTAIN,ethnicity_UNKNOWN,ethnicity_WHITE,gender_F,gender_M
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0


In [24]:
#One hot encoding of cat cols

#X_test = pd.get_dummies(X_test, columns=test_cat_cols, drop_first=True)
#X_test = X_test.reindex(columns=df.columns, fill_value=0)



In [71]:
#x = [c for c in df.columns if c not in X_test.columns]
#print(x)
#df.info()
#print(f"df columns: {df}")
#print(f"test columns: {X_test.columns}")
a = X_test.select_dtypes(include=['bool']).columns
b = df.select_dtypes(include=['bool']).columns
print(a)
print("-"*40)

x = [c for c in num_cols if c not in test_num_cols]
print(x)

Index([], dtype='object')
----------------------------------------
[]


In [72]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer

imp = IterativeImputer(random_state=42, max_iter=10)
df_imputed = imp.fit_transform(df[num_cols])

In [73]:
X_test = X_test.reindex(columns=num_cols)
test_imputed = imp.transform(X_test[num_cols])

In [74]:
df_imputed = pd.DataFrame(df_imputed, columns=num_cols, index=df.index)
test_imputed = pd.DataFrame(test_imputed, columns=test_num_cols, index=X_test.index)
test_imputed.head()

,los,age,pvd,htn,copd,diabetes,renal,liver,pud,sepsis,...,mean_diastolic_BP,mean_mean_BP,mean_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score
0,7.1280,75.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,96.011915,126.357552,98.584348,70.094090,95.927708,49.048108,61.163887,92.626062,20.0,1.0
1,14.0641,64.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,...,83.000000,117.000000,100.000000,66.000000,84.000000,38.000000,48.000000,100.000000,36.0,3.0
2,2.6538,80.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,...,106.265035,123.896246,92.805074,68.570937,97.218249,45.706620,59.979800,90.624443,12.0,4.0
3,3.8344,67.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,74.000000,94.000000,100.000000,73.000000,84.000000,37.000000,52.000000,94.000000,9.0,1.0
4,19.6682,73.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,104.495528,127.494588,82.497223,72.313692,91.408244,48.596507,59.732073,94.524178,0.0,0.0


In [75]:
num_st = [c for c in num_cols if c not in ['pvd', 'htn', 'copd', 'diabetes',
                          'renal', 'liver', 'pud', 'sepsis', 'chf', 'cancer',
                          'rheumatic', 'Alcohol', 'Tobacco', 'Opioid',
                          'Cannabis', 'Cocaine', 'Sedative', 'Other', 'Smoking'] ]

#num_st

In [76]:
from sklearn.preprocessing import StandardScaler

sc = StandardScaler()
df_scaled = sc.fit_transform(df_imputed[num_st])
test_scaled = sc.transform(test_imputed[num_st])


In [77]:
df_imputed[num_st] = df_scaled
test_imputed[num_st] = test_scaled
df_imputed.head()

,los,age,pvd,htn,copd,diabetes,renal,liver,pud,sepsis,...,max_diastolic_BP,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score
0,-0.083607,-1.206413,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.052305,-0.030983,-0.008440,-0.724344,0.064523,0.638435,0.238671,0.440757,-0.731203,-0.826270
1,-0.003261,0.403019,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,...,-0.040903,-0.021330,-0.024971,-0.525181,1.306274,0.121995,0.319263,0.288853,-0.731203,-0.826270
2,-0.164206,-0.354361,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,-0.027428,-0.022709,-0.016705,-0.458794,-0.853292,-1.255179,-0.970205,0.288853,-0.484406,1.963532
3,-0.497217,-0.827723,1.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.001878,-0.001970,-0.008440,0.537019,0.077202,0.172049,0.120167,-0.014954,1.489976,-0.826270
4,0.000354,-0.827723,0.0,1.0,1.0,1.0,0.0,0.0,0.0,0.0,...,-0.018099,-0.015125,-0.008440,1.665606,-0.907281,0.035921,0.238671,0.136949,1.613375,-0.826270


In [78]:
df_imputed.columns

Index(['los', 'age', 'pvd', 'htn', 'copd', 'diabetes', 'renal', 'liver', 'pud',
       'sepsis', 'chf', 'cancer', 'rheumatic', 'Alcohol', 'Tobacco', 'Opioid',
       'Cannabis', 'Cocaine', 'Sedative', 'Other', 'Smoking', 'mean_Anion Gap',
       'mean_Ca', 'mean_chloride', 'mean_Creatinine', 'mean_Glucose',
       'mean_Mg', 'mean_Phosphate', 'mean_Potassium', 'mean_Na',
       'mean_Urea Nitrogen', 'mean_INR(PT)', 'mean_Platelet Count', 'mean_PT',
       'mean_PTT', 'mean_WBC', 'max_Anion Gap', 'max_Ca', 'max_chloride',
       'max_Creatinine', 'max_Glucose', 'max_Mg', 'max_Phosphate',
       'max_Potassium', 'max_Na', 'max_Urea Nitrogen', 'max_INR(PT)',
       'max_Platelet Count', 'max_PT', 'max_PTT', 'max_WBC', 'min_Anion Gap',
       'min_Ca', 'min_chloride', 'min_Creatinine', 'min_Glucose', 'min_Mg',
       'min_Phosphate', 'min_Potassium', 'min_Na', 'min_Urea Nitrogen',
       'min_INR(PT)', 'min_Platelet Count', 'min_PT', 'min_PTT', 'min_WBC',
       'mean_Heart Rate', 'mean_sy

In [79]:
df_encoded.loc[:, num_cols] = df_imputed.values
X_test_encoded.loc[:, num_cols]  = test_imputed.values

df_encoded.head()

,first_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Cardiac Vascular Intensive Care Unit (CVICU),last_careunit_Coronary Care Unit (CCU),last_careunit_Medical Intensive Care Unit (MICU),last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),last_careunit_Neuro Intermediate,last_careunit_Neuro Stepdown,last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),last_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Trauma SICU (TSICU),...,max_diastolic_BP,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.052305,-0.030983,-0.008440,-0.724344,0.064523,0.638435,0.238671,0.440757,-0.731203,-0.826270
1,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.040903,-0.021330,-0.024971,-0.525181,1.306274,0.121995,0.319263,0.288853,-0.731203,-0.826270
2,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.027428,-0.022709,-0.016705,-0.458794,-0.853292,-1.255179,-0.970205,0.288853,-0.484406,1.963532
3,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.001878,-0.001970,-0.008440,0.537019,0.077202,0.172049,0.120167,-0.014954,1.489976,-0.826270
4,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.018099,-0.015125,-0.008440,1.665606,-0.907281,0.035921,0.238671,0.136949,1.613375,-0.826270


In [80]:
X_test_encoded.head()

,first_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Cardiac Vascular Intensive Care Unit (CVICU),last_careunit_Coronary Care Unit (CCU),last_careunit_Medical Intensive Care Unit (MICU),last_careunit_Medical/Surgical Intensive Care Unit (MICU/SICU),last_careunit_Neuro Intermediate,last_careunit_Neuro Stepdown,last_careunit_Neuro Surgical Intensive Care Unit (Neuro SICU),last_careunit_Surgical Intensive Care Unit (SICU),last_careunit_Trauma SICU (TSICU),...,max_diastolic_BP,max_mean_BP,max_Spo2,min_Heart Rate,min_systolic_BP,min_diastolic_BP,min_mean_BP,min_Spo2,elix_score,SOFA_score
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.818219,0.769714,0.496166,0.144940,-0.155336,0.040062,-0.070488,0.080147,1.489976,-0.128819
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.477280,0.618316,0.932401,-0.126856,-0.799303,-0.910886,-1.131389,1.200275,3.464359,1.266082
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.086871,0.729892,-1.284725,0.043822,-0.085661,-0.247551,-0.165916,-0.223906,0.502785,1.963532
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.241462,0.246193,0.932401,0.337856,-0.799303,-0.996959,-0.809022,0.288853,0.132589,-0.128819
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.040506,0.788111,-4.461103,0.292294,-0.399338,0.001191,-0.185881,0.368478,-0.978001,-0.826270


In [35]:
#X_test.columns

In [81]:
y.value_counts()

is_stroke
0    5621
1    1255
Name: count, dtype: int64

In [82]:
df = df_encoded.copy()
X_test = X_test_encoded.copy()

len(X_test.columns)

106

In [83]:
df = df.merge(y, how='left', on=df.index)
test = X_test.merge(y_test, how='left', on=X_test.index)

In [84]:
df.to_csv('mimic_iv_after_preprocessed_train.csv')
test.to_csv('mimic_iii_after_preprocessed_test.csv')